In [8]:
# importing libraries
import os
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"

In [10]:
# loading gallery database
def load_gallery_db(gallery_root):

    gallery_db = {}

    for identity in os.listdir(gallery_root):

        gallery_db[identity] = []

        identity_dir = os.path.join(gallery_root, identity)

        for file in os.listdir(identity_dir):

            emb = np.load(os.path.join(identity_dir, file))

            gallery_db[identity].append(emb)

    return gallery_db

In [11]:
# finding hardest negative
def get_hard_negative(probe_embedding, true_identity, gallery_db):

    scores = {}

    for identity in gallery_db:

        if identity == true_identity:
            continue

        sims = []

        for gallery_emb in gallery_db[identity]:

            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]

            sims.append(sim)

        scores[identity] = max(sims)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    hard_negative_identity = sorted_scores[0][0]

    hard_negative_similarity = sorted_scores[0][1]

    second_similarity = sorted_scores[1][1]

    margin = hard_negative_similarity - second_similarity

    return (hard_negative_identity, hard_negative_similarity, margin)

In [12]:
# resuing musiq
import torch
import pyiqa

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


In [13]:
# get quality score
def get_quality_score(image_path):

    score = musiq_metric(image_path)

    return float(score.cpu().item())

In [14]:
# generating hard negatives
gallery_db = load_gallery_db(
    os.path.join(ROOT_EMBEDDINGS, "train", "gallery")
)

rows = []

probe_emb_root = os.path.join(ROOT_EMBEDDINGS, "train", "degraded_probes")

probe_img_root = os.path.join(ROOT_IMAGES, "train", "degraded_probes")

counter = 0

for identity in tqdm(os.listdir(probe_emb_root)):

    emb_dir = os.path.join(probe_emb_root, identity)

    img_dir = os.path.join(probe_img_root, identity)

    for emb_file in os.listdir(emb_dir):

        probe_embedding = np.load(os.path.join(emb_dir, emb_file))

        base_name = os.path.splitext(emb_file)[0]

        image_path = None

        for ext in [".jpg", ".jpeg", ".png"]:

            candidate = os.path.join(img_dir, base_name + ext)

            if os.path.exists(candidate):

                image_path = candidate
                break

        if image_path is None:
            continue

        quality_score = get_quality_score(image_path)

        negative_identity, similarity, margin = get_hard_negative(
            probe_embedding, identity, gallery_db
        )

        rows.append(
            {
                "quality_score": quality_score,
                "best_similarity": similarity,
                "margin": margin,
                "true_identity": identity,
                "negative_identity": negative_identity,
                "label": 0,
            }
        )

        counter += 1

        if counter % 100 == 0:

            print(f"Generated {counter}")

            print("True:", identity)

            print("Hard Negative:", negative_identity)

            print("Similarity:", round(similarity, 4))

            print("Margin:", round(margin, 4))

  9%|▉         | 9/100 [00:41<07:16,  4.80s/it]

Generated 100
True: Bill_McBride
Hard Negative: Gerhard_Schroeder
Similarity: 0.2532
Margin: 0.0197


 15%|█▌        | 15/100 [01:21<07:36,  5.37s/it]

Generated 200
True: Taha_Yassin_Ramadan
Hard Negative: Jean_Chretien
Similarity: 0.3047
Margin: 0.0798


 21%|██        | 21/100 [01:57<07:47,  5.92s/it]

...

 92%|█████████▏| 92/100 [13:15<00:55,  6.95s/it]

Generated 2000
True: Atal_Bihari_Vajpayee
Hard Negative: Mahmoud_Abbas
Similarity: 0.2684
Margin: 0.0689


 96%|█████████▌| 96/100 [13:55<00:36,  9.02s/it]

Generated 2100
True: Gray_Davis
Hard Negative: John_Ashcroft
Similarity: 0.3214
Margin: 0.0631


100%|██████████| 100/100 [14:32<00:00,  8.72s/it]


In [15]:
# saving hard negatives
hard_negative_df = pd.DataFrame(rows)

# hard negative similarity stats
print("\nHard Negative Similarity Statistics")

print(hard_negative_df["best_similarity"].describe())

print("\nHard Negative Margin Statistics")

print(hard_negative_df["margin"].describe())

hard_negative_df.to_csv("hard_negative_dataset_debug.csv", index=False)
hard_negative_train = hard_negative_df[
    ["quality_score", "best_similarity", "margin", "label"]
]

hard_negative_train.to_csv("hard_negative_dataset.csv", index=False)
print(hard_negative_df.shape)

print(hard_negative_df["label"].value_counts())


Hard Negative Similarity Statistics
count    2196.000000
mean        0.266360
std         0.055582
min         0.158531
25%         0.229096
50%         0.258467
75%         0.291716
max         0.718965
Name: best_similarity, dtype: float64

Hard Negative Margin Statistics
count    2196.000000
mean        0.033406
std         0.040255
min         0.000008
25%         0.008495
50%         0.021581
75%         0.042716
max         0.375892
Name: margin, dtype: float64
(2196, 6)
label
0    2196
Name: count, dtype: int64


In [17]:
# combining withe existing dataset
original_df = pd.read_csv("/Users/admin/Desktop/reliable_rejection_under_degradation/second_half/embeddings/train_svm.csv")

extended_df = pd.concat([original_df, hard_negative_train], ignore_index=True)

# checking balance
print("\nFinal Label Distribution")

print(extended_df["label"].value_counts())

print("\nFinal Label Percentages")

print(extended_df["label"].value_counts(normalize=True) * 100)

extended_df.to_csv("train_svm_extended.csv", index=False)

print(extended_df.shape)

print(extended_df["label"].value_counts())


Final Label Distribution
label
0    2237
1    2155
Name: count, dtype: int64

Final Label Percentages
label
0    50.933515
1    49.066485
Name: proportion, dtype: float64
(4392, 4)
label
0    2237
1    2155
Name: count, dtype: int64


In [18]:
# sanity check
print("\n===== HARD NEGATIVE SIMILARITY =====")

print(hard_negative_df["best_similarity"].describe())

print("\n===== HARD NEGATIVE MARGIN =====")

print(hard_negative_df["margin"].describe())

print("\n===== FINAL LABEL DISTRIBUTION =====")

print(extended_df["label"].value_counts())

print("\n===== FINAL LABEL PERCENTAGES =====")

print(extended_df["label"].value_counts(normalize=True) * 100)


===== HARD NEGATIVE SIMILARITY =====
count    2196.000000
mean        0.266360
std         0.055582
min         0.158531
25%         0.229096
50%         0.258467
75%         0.291716
max         0.718965
Name: best_similarity, dtype: float64

===== HARD NEGATIVE MARGIN =====
count    2196.000000
mean        0.033406
std         0.040255
min         0.000008
25%         0.008495
50%         0.021581
75%         0.042716
max         0.375892
Name: margin, dtype: float64

===== FINAL LABEL DISTRIBUTION =====
label
0    2237
1    2155
Name: count, dtype: int64

===== FINAL LABEL PERCENTAGES =====
label
0    50.933515
1    49.066485
Name: proportion, dtype: float64
